In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
df = pd.read_csv('/content/drive/MyDrive/mlops/tweets.csv')
display(df.head())

,id,keyword,location,text,target
0,0,ablaze,NaN,"Communal violence in Bhainsa, Telangana. ""Ston...",1
1,1,ablaze,NaN,Telangana: Section 144 has been imposed in Bha...,1
2,2,ablaze,New York City,Arsonist sets cars ablaze at dealership https:...,1
3,3,ablaze,"Morgantown, WV",Arsonist sets cars ablaze at dealership https:...,1
4,4,ablaze,NaN,"""Lord Jesus, your love brings freedom and pard...",0


In [ ]:
df = pd.read_csv('/content/drive/MyDrive/mlops/tweets.csv')

df.head()

,id,keyword,location,text,target
0,0,ablaze,NaN,"Communal violence in Bhainsa, Telangana. ""Ston...",1
1,1,ablaze,NaN,Telangana: Section 144 has been imposed in Bha...,1
2,2,ablaze,New York City,Arsonist sets cars ablaze at dealership https:...,1
3,3,ablaze,"Morgantown, WV",Arsonist sets cars ablaze at dealership https:...,1
4,4,ablaze,NaN,"""Lord Jesus, your love brings freedom and pard...",0


In [ ]:
df = df[['keyword', 'text', 'target']]

df.head()

,keyword,text,target
0,ablaze,"Communal violence in Bhainsa, Telangana. ""Ston...",1
1,ablaze,Telangana: Section 144 has been imposed in Bha...,1
2,ablaze,Arsonist sets cars ablaze at dealership https:...,1
3,ablaze,Arsonist sets cars ablaze at dealership https:...,1
4,ablaze,"""Lord Jesus, your love brings freedom and pard...",0


In [ ]:
df['keyword'] = df['keyword'].fillna('')

df['text'] = df['text'].fillna('')

In [ ]:
df.drop_duplicates(inplace=True)

print(df.shape)

(11228, 3)


,text,target,clean_text
0,"Communal violence in Bhainsa, Telangana. ""Ston...",1,communal violence in bhainsa telangana stones ...
1,Telangana: Section 144 has been imposed in Bha...,1,telangana section has been imposed in bhainsa...
2,Arsonist sets cars ablaze at dealership https:...,1,arsonist sets cars ablaze at dealership
3,Arsonist sets cars ablaze at dealership https:...,1,arsonist sets cars ablaze at dealership
4,"""Lord Jesus, your love brings freedom and pard...",0,lord jesus your love brings freedom and pardon...


In [ ]:
df['combined_text'] = df['keyword'] + " " + df['text']

df.head()

,keyword,text,target,combined_text
0,ablaze,"Communal violence in Bhainsa, Telangana. ""Ston...",1,"ablaze Communal violence in Bhainsa, Telangana..."
1,ablaze,Telangana: Section 144 has been imposed in Bha...,1,ablaze Telangana: Section 144 has been imposed...
2,ablaze,Arsonist sets cars ablaze at dealership https:...,1,ablaze Arsonist sets cars ablaze at dealership...
3,ablaze,Arsonist sets cars ablaze at dealership https:...,1,ablaze Arsonist sets cars ablaze at dealership...
4,ablaze,"""Lord Jesus, your love brings freedom and pard...",0,"ablaze ""Lord Jesus, your love brings freedom a..."


In [ ]:
import re

def clean_text(text):

    text = text.lower()

    text = re.sub(r"http\S+", "", text)

    text = re.sub(r"@\w+", "", text)

    text = re.sub(r"#", "", text)

    text = re.sub(r"[^a-zA-Z\s]", "", text)

    return text

In [ ]:
df['clean_text'] = df['combined_text'].apply(clean_text)

df.head()

,keyword,text,target,combined_text,clean_text
0,ablaze,"Communal violence in Bhainsa, Telangana. ""Ston...",1,"ablaze Communal violence in Bhainsa, Telangana...",ablaze communal violence in bhainsa telangana ...
1,ablaze,Telangana: Section 144 has been imposed in Bha...,1,ablaze Telangana: Section 144 has been imposed...,ablaze telangana section has been imposed in ...
2,ablaze,Arsonist sets cars ablaze at dealership https:...,1,ablaze Arsonist sets cars ablaze at dealership...,ablaze arsonist sets cars ablaze at dealership
3,ablaze,Arsonist sets cars ablaze at dealership https:...,1,ablaze Arsonist sets cars ablaze at dealership...,ablaze arsonist sets cars ablaze at dealership
4,ablaze,"""Lord Jesus, your love brings freedom and pard...",0,"ablaze ""Lord Jesus, your love brings freedom a...",ablaze lord jesus your love brings freedom and...


In [ ]:
X = df['clean_text']

y = df['target']

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer()

X = tfidf.fit_transform(X)

print(X.shape)

(11228, 21770)


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [ ]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()

model.fit(X_train, y_train)

LinearRegression()

In [ ]:
pred = model.predict(X_test)

print(pred[:10])

[ 0.55012175  0.67577513 -0.72936646 -1.19688586  1.32568979 -3.15124618
 -0.72382926  0.01972136  0.33064464  2.5631096 ]


In [ ]:
pred_binary = []

for i in pred:

    if i >= 0.5:
        pred_binary.append(1)

    else:
        pred_binary.append(0)

In [ ]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, pred_binary)

print("Accuracy:", accuracy)

Accuracy: 0.6945681211041852


In [ ]:
sample = ["earthquake destroyed buildings"]

sample_vector = tfidf.transform(sample)

result = model.predict(sample_vector)

print(result)

[0.76296206]


In [ ]:
if result[0] >= 0.5:
    print("Disaster Tweet")

else:
    print("Not Disaster Tweet")

Disaster Tweet


In [ ]:
import pickle

with open("multivariate_model.pkl", "wb") as file:
    pickle.dump(model, file)

with open("tfidf.pkl", "wb") as file:
    pickle.dump(tfidf, file)

print("Pickle files saved")

NameError: name 'model' is not defined

In [ ]:
import joblib

joblib.dump(model, "multivariate_model.joblib")

joblib.dump(tfidf, "tfidf.joblib")

print("Joblib files saved")

Joblib files saved
